In [9]:
import pandas as pd
import seaborn as sns
sns.set()
import mlflow
import subprocess

from pathlib import Path

from qrt.data import load_tr_data
from qrt.splitters import make_date_folds
from qrt.train import train_and_predict_cv, predict_positive_probability
from qrt.features import compute_features

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score

In [10]:
X_tr, y_tr = load_tr_data(Path("../data/raw"))

In [11]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("qrt-aapf")

cv_params = {"n_splits": 8, "seed": 0}
folds = make_date_folds(X_tr, **cv_params)

X = X_tr.drop(columns=["TS", "GROUP", "ALLOCATION"])
X = compute_features(X)
y = (y_tr["target"] > 0).astype(int)

model = LGBMClassifier(
    n_estimators=200,
    num_leaves=7,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1,
)


In [12]:
root = Path("..").resolve()

with mlflow.start_run(run_name="lgbm_fts_v2_seed0") as run:
    # Record the experiment configuration
    mlflow.set_tags({
        "model": type(model).__name__,
        "reason": "Test seed cv stability",
        "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=root, text=True
        ).strip(),
    })

    mlflow.log_params(model.get_params())
    mlflow.log_params({
        "cv_n_splits": cv_params["n_splits"],
        "cv_seed": cv_params["seed"],
        "prediction_threshold": 0.5,
    })

    mlflow.log_dict({
        "features": X.columns.tolist(),
        "n_features": X.shape[1],
        "target": "target > 0",
        "training_files": ["X_train.csv", "y_train.csv"],
        "cv_method": "KFold on shuffled unique TS",
    }, "config.json")

    # Save the source files, including uncommitted changes
    for path in (root / "src/qrt").glob("*.py"):
        mlflow.log_artifact(str(path), artifact_path="source/qrt")

    for name in (
        "pyproject.toml",
        "uv.lock",
        "notebooks/02_models.ipynb",
    ):
        mlflow.log_artifact(str(root / name), artifact_path="source")

    # Train and evaluate
    oof = train_and_predict_cv(model, X, y, folds, predict_positive_probability)
    assert oof.notna().all()

    y_pred = (oof >= 0.5).astype(int)
    scores = {"accuracy": accuracy_score(y, y_pred)}
    for fold in sorted(folds.unique()):
        rows = folds == fold
        scores[f"accuracy_fold_{fold}"] = accuracy_score(y[rows], y_pred[rows])

    mlflow.log_metrics(scores)

    # Save predictions and the exact fold assignments
    predictions = pd.DataFrame({
        "fold": folds,
        "target": y,
        "prediction": oof,
    })
    mlflow.log_text(predictions.to_csv(), "oof.csv")

pd.Series(scores)

🏃 View run lgbm_fts_v2_seed0 at: http://127.0.0.1:5001/#/experiments/2/runs/c39be040c4414e4491033555d33cd1fd
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2


accuracy           0.523159
accuracy_fold_0    0.524940
accuracy_fold_1    0.528945
accuracy_fold_2    0.523210
accuracy_fold_3    0.525920
accuracy_fold_4    0.516918
accuracy_fold_5    0.523758
accuracy_fold_6    0.519793
accuracy_fold_7    0.521947
dtype: float64